In [ ]:

import random

suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']
deck = [(rank, suit) for suit in suits for rank in ranks]

trials = 100000
red_count = 0

for _ in range(trials):
    card = random.choice(deck)
    if card[1] in ['Hearts', 'Diamonds']:
        red_count += 1

print("Theoretical P(Red):", 26/52)
print("Simulated P(Red):", red_count/trials)


red_cards = [card for card in deck if card[1] in ['Hearts', 'Diamonds']]

heart_count = 0

for _ in range(trials):
    card = random.choice(red_cards)
    if card[1] == 'Hearts':
        heart_count += 1

print("Theoretical P(Heart | Red):", 13/26)
print("Simulated P(Heart | Red):", heart_count/trials)

face_cards = [card for card in deck if card[0] in ['Jack', 'Queen', 'King']]

diamond_count = 0

for _ in range(trials):
    card = random.choice(face_cards)
    if card[1] == 'Diamonds':
        diamond_count += 1

print("Theoretical P(Diamond | Face):", 3/12)
print("Simulated P(Diamond | Face):", diamond_count/trials)

count = 0

for _ in range(trials):
    card = random.choice(face_cards)
    if card[1] == 'Spades' or card[0] == 'Queen':
        count += 1

print("Theoretical P(Spade OR Queen | Face):", 6/12)
print("Simulated P(Spade OR Queen | Face):", count/trials)

Theoretical P(Red): 0.5
Simulated P(Red): 0.50042
Theoretical P(Heart | Red): 0.5
Simulated P(Heart | Red): 0.5032
Theoretical P(Diamond | Face): 0.25
Simulated P(Diamond | Face): 0.24938
Theoretical P(Spade OR Queen | Face): 0.5
Simulated P(Spade OR Queen | Face): 0.50022


In [2]:
pip install pgmpy

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 479.2 kB/s eta 0:00:04
   -------- ------------------------------- 0.5/2.4 MB 479.2 kB/s eta 0:00:04
   ------------- -------------------------- 0.8/2.4 MB 508.5 kB/s eta 0:00:04
   ------------- -------------------------- 0.8/2.4 MB 508.5 kB/s eta 0:00:04
   ------------- -------------------------- 0.8/2.4 MB 508.5 kB/s eta 0:00:04
   ----------------- ---------------------- 1.0/2.4 MB 513.7 kB/s eta 0:00:03
   ----------------- ---------------------- 1.0/2.4 MB 513.7 kB/s eta 0:00:03
   --------------------- ---------------


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:


from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('I', 'G'),
    ('S', 'G'),
    ('D', 'G'),
    ('G', 'P')
])

cpd_I = TabularCPD(
    variable='I', variable_card=2,
    values=[[0.7], [0.3]],
    state_names={'I': ['High', 'Low']}
)

cpd_S = TabularCPD(
    variable='S', variable_card=2,
    values=[[0.6], [0.4]],
    state_names={'S': ['Sufficient', 'Insufficient']}
)

cpd_D = TabularCPD(
    variable='D', variable_card=2,
    values=[[0.4], [0.6]],
    state_names={'D': ['Hard', 'Easy']}
)


cpd_G = TabularCPD(
    variable='G', variable_card=3,
    values=[
        [0.7, 0.5, 0.4, 0.3, 0.5, 0.3, 0.2, 0.1], #A
        [0.2, 0.3, 0.4, 0.4, 0.3, 0.4, 0.4, 0.3], #B
        [0.1, 0.2, 0.2, 0.3, 0.2, 0.3, 0.4, 0.6] #C
    ],
    evidence=['I', 'S', 'D'],
    evidence_card=[2, 2, 2],
    state_names={
        'G': ['A', 'B', 'C'],
        'I': ['High', 'Low'],
        'S': ['Sufficient', 'Insufficient'],
        'D': ['Hard', 'Easy']
    }
)

cpd_P = TabularCPD(
    variable='P', variable_card=2,
    values=[
        [0.95, 0.80, 0.50],  # Yes
        [0.05, 0.20, 0.50]   # No
    ],
    evidence=['G'],
    evidence_card=[3],
    state_names={
        'P': ['Yes', 'No'],
        'G': ['A', 'B', 'C']
    }
)

model.add_cpds(cpd_I, cpd_S, cpd_D, cpd_G, cpd_P)
assert model.check_model()
infer = VariableElimination(model)

q1 = infer.query(
    variables=['P'],
    evidence={'S': 'Sufficient', 'D': 'Hard'}
)
print("P(Pass | S=Sufficient, D=Hard):")
print(q1)

q2 = infer.query(
    variables=['I'],
    evidence={'P': 'Yes'}
)
print("\nP(Intelligence | Pass=Yes):")
print(q2)


P(Pass | S=Sufficient, D=Hard):
+--------+----------+
| P      |   phi(P) |
+========+==========+
| P(Yes) |   0.8570 |
+--------+----------+
| P(No)  |   0.1430 |
+--------+----------+

P(Intelligence | Pass=Yes):
+---------+----------+
| I       |   phi(I) |
+=========+==========+
| I(High) |   0.7211 |
+---------+----------+
| I(Low)  |   0.2789 |
+---------+----------+


In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'),
    ('Disease', 'Cough'),
    ('Disease', 'Fatigue'),
    ('Disease', 'Chills')
])

cpd_D = TabularCPD(
    variable='Disease', variable_card=2,
    values=[[0.3], [0.7]],
    state_names={'Disease': ['Flu', 'Cold']}
)

cpd_Fever = TabularCPD(
    variable='Fever', variable_card=2,
    values=[
        [0.9, 0.5],  
        [0.1, 0.5]   
    ],
    evidence=['Disease'],
    evidence_card=[2],
    state_names={'Fever': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

cpd_Cough = TabularCPD(
    variable='Cough', variable_card=2,
    values=[
        [0.8, 0.6],
        [0.2, 0.4]
    ],
    evidence=['Disease'],
    evidence_card=[2],
    state_names={'Cough': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

cpd_Fatigue = TabularCPD(
    variable='Fatigue', variable_card=2,
    values=[
        [0.7, 0.3],
        [0.3, 0.7]
    ],
    evidence=['Disease'],
    evidence_card=[2],
    state_names={'Fatigue': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

cpd_Chills = TabularCPD(
    variable='Chills', variable_card=2,
    values=[
        [0.6, 0.4],
        [0.4, 0.6]
    ],
    evidence=['Disease'],
    evidence_card=[2],
    state_names={'Chills': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

model.add_cpds(cpd_D, cpd_Fever, cpd_Cough, cpd_Fatigue, cpd_Chills)
assert model.check_model()
infer = VariableElimination(model)

q1 = infer.query(
    variables=['Disease'],
    evidence={'Fever': 'Yes', 'Cough': 'Yes'}
)
print("P(Disease | Fever=Yes, Cough=Yes):")
print(q1)

q2 = infer.query(
    variables=['Disease'],
    evidence={'Fever': 'Yes', 'Cough': 'Yes', 'Chills': 'Yes'}
)
print("\nP(Disease | Fever=Yes, Cough=Yes, Chills=Yes):")
print(q2)

q3 = infer.query(
    variables=['Fatigue'],
    evidence={'Disease': 'Flu'}
)
print("\nP(Fatigue | Disease=Flu):")
print(q3)


P(Disease | Fever=Yes, Cough=Yes):
+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.5070 |
+---------------+----------------+
| Disease(Cold) |         0.4930 |
+---------------+----------------+

P(Disease | Fever=Yes, Cough=Yes, Chills=Yes):
+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.6067 |
+---------------+----------------+
| Disease(Cold) |         0.3933 |
+---------------+----------------+

P(Fatigue | Disease=Flu):
+--------------+----------------+
| Fatigue      |   phi(Fatigue) |
+==============+================+
| Fatigue(Yes) |         0.7000 |
+--------------+----------------+
| Fatigue(No)  |         0.3000 |
+--------------+----------------+


In [13]:
import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]
P = np.array([
    #sunny, cloudy, rainy
    [0.6, 0.3, 0.1],  # Sunny
    [0.3, 0.4, 0.3],  # Cloudy
    [0.2, 0.3, 0.5]   # Rainy
])

def simulate_weather(initial_state, days):
    current = states.index(initial_state)
    sequence = [initial_state]
    rainy_days = 0

    for _ in range(days):
        next_state = np.random.choice(states, p=P[current])
        sequence.append(next_state)

        if next_state == "Rainy":
            rainy_days += 1

        current = states.index(next_state)

    return sequence, rainy_days

sequence, rainy_days = simulate_weather("Sunny", 10)

print("Weather sequence:")
print(" -> ".join(sequence))
print("Rainy days:", rainy_days)
trials = 10000
count = 0

for _ in range(trials):
    _, rainy = simulate_weather("Sunny", 10)
    
    if rainy >= 3:
        count += 1

probability = count / trials

print("Probability of at least 3 rainy days:", probability)


Weather sequence:
Sunny -> Sunny -> Sunny -> Sunny -> Cloudy -> Sunny -> Sunny -> Sunny -> Sunny -> Sunny -> Rainy
Rainy days: 1
Probability of at least 3 rainy days: 0.4559
